# Pipeline 3 — Multi-branch stacking

This notebook implements the extended system:

- **Task 1:** XLM-RoBERTa branch + TF-IDF Logistic Regression branch.
- **Task 2:** XLM-RoBERTa branch + perplexity/stylometry GBDT branch.

The final probability is learned by a small Logistic Regression meta-learner on out-of-fold branch probabilities, followed by threshold tuning for positive-class F1.

In [ ]:
# Kaggle usually already has most packages. Run this cell if imports fail.
# !pip -q install -r ../requirements.txt
# !pip -q install transformers accelerate sentencepiece scikit-learn tqdm

In [ ]:
# ============================================================
# Configuration
# ============================================================
from pathlib import Path

PIPELINE_NAME = "Pipeline 3 — Multi-branch stacking"
PIPELINE_SLUG = "pipeline3_multibranch"
MODEL_NAME = "xlm-roberta-base"
MODEL_DISPLAY = "XLM-RoBERTa-base"

TASKS_TO_RUN = [1, 2]
SEED = 42
N_FOLDS = 5
FAST_DEBUG = False
DATA_DIR = None
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Perplexity branch is useful but slower because it downloads/runs a Spanish GPT-2 model.
# If Kaggle time is limited, set this to False; stylometry features will still run.
USE_PERPLEXITY = True
PERPLEXITY_MODEL_NAME = "datificate/gpt2-small-spanish"

BASE_TASK_CONFIGS = {
    1: {
        "name": "Subtask 1 — Satirical News Detection",
        "positive_label": "satirical",
        "label2id": {"real": 0, "satirical": 1},
        "id2label": {0: "real", 1: "satirical"},
        "train_files": ["task1_trial.tsv", "task1_dev_gold.tsv"],
        "test_file": "task1_test.tsv",
        "max_length": 128,
        "batch_size": 8,
        "epochs": 5,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.06,
        "weight_decay": 0.01,
        "gradient_checkpointing": True,
    },
    2: {
        "name": "Subtask 2 — LLM Humor Detection",
        "positive_label": "machine",
        "label2id": {"human": 0, "machine": 1},
        "id2label": {0: "human", 1: "machine"},
        "train_files": ["task2_trial.tsv", "task2_dev_gold.tsv"],
        "test_file": "task2_test.tsv",
        "max_length": 192,
        "batch_size": 8,
        "epochs": 4,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.06,
        "weight_decay": 0.01,
        "gradient_checkpointing": True,
    },
}

print(f"Pipeline: {PIPELINE_NAME}")
print(f"Transformer branch: {MODEL_DISPLAY} ({MODEL_NAME})")
print(f"Output: {OUTPUT_ROOT.resolve()}")

In [ ]:
# ============================================================
# Imports and reproducibility
# ============================================================
import os
import gc
import re
import zipfile
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()
print("Device:", DEVICE)
print("FP16 enabled:", USE_FP16)

In [ ]:
# ============================================================
# Dataset discovery and preprocessing
# ============================================================
REQUIRED_DATA_FILES = {
    "task1_trial.tsv", "task1_dev_gold.tsv", "task1_test.tsv",
    "task2_trial.tsv", "task2_dev_gold.tsv", "task2_test.tsv",
}

def find_dataset_dir(user_dir=None) -> Path:
    if user_dir is not None:
        p = Path(user_dir)
        if p.exists():
            return p
        raise FileNotFoundError(f"DATA_DIR was set but does not exist: {p}")

    candidates = [
        Path("./dataset"),
        Path("../dataset"),
        Path("/kaggle/working/dataset"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend([p for p in kaggle_input.rglob("*") if p.is_dir()])

    for p in candidates:
        if p.exists() and REQUIRED_DATA_FILES.issubset({x.name for x in p.glob("*.tsv")}):
            return p

    raise FileNotFoundError(
        "Could not find dataset directory. Set DATA_DIR to the folder containing the task TSV files."
    )

DATA_PATH = find_dataset_dir(DATA_DIR)
print("Dataset directory:", DATA_PATH.resolve())


def clean_text(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_text(row, task_id: int) -> str:
    headline = clean_text(row.get("headline", ""))
    if task_id == 1:
        context = clean_text(row.get("context", ""))
        return headline if not context else f"{headline} [SEP] {context}"
    if task_id == 2:
        joke = clean_text(row.get("joke", ""))
        hint = "[LONG]" if len(joke.split()) >= 18 else "[SHORT]"
        return f"{headline} [SEP] {hint} {joke}"
    raise ValueError(f"Unsupported task_id: {task_id}")


def load_task_data(task_id: int):
    cfg = BASE_TASK_CONFIGS[task_id]
    frames = []
    for filename in cfg["train_files"]:
        path = DATA_PATH / filename
        df_part = pd.read_csv(path, sep="\t")
        df_part["source_file"] = filename
        frames.append(df_part)
    train_df = pd.concat(frames, ignore_index=True)
    test_df = pd.read_csv(DATA_PATH / cfg["test_file"], sep="\t")

    if "tag" not in train_df.columns:
        raise ValueError(f"Missing 'tag' column for task {task_id}")
    train_df["label"] = train_df["tag"].map(cfg["label2id"])
    if train_df["label"].isna().any():
        bad = train_df.loc[train_df["label"].isna(), "tag"].unique().tolist()
        raise ValueError(f"Unknown labels in task {task_id}: {bad}")
    train_df["label"] = train_df["label"].astype(int)

    train_df["text"] = train_df.apply(lambda r: build_text(r, task_id), axis=1)
    test_df["text"] = test_df.apply(lambda r: build_text(r, task_id), axis=1)

    if FAST_DEBUG:
        train_df = train_df.groupby("label", group_keys=False).head(24).reset_index(drop=True)
        test_df = test_df.head(16).reset_index(drop=True)
        cfg = dict(cfg)
        cfg["epochs"] = 1
        cfg["batch_size"] = min(8, cfg["batch_size"])

    print(f"Task {task_id}: train={train_df.shape}, test={test_df.shape}")
    print(train_df["tag"].value_counts().to_dict())
    return train_df, test_df, cfg

for task_id in TASKS_TO_RUN:
    _train_df, _test_df, _cfg = load_task_data(task_id)
    display(_train_df[["id", "tag", "text"]].head(3))

In [ ]:
# ============================================================
# Modeling utilities
# ============================================================
class TextClassificationDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length, labels=None):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def prepare_tokenizer(task_id: int):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    desired_special_tokens = ["[SEP]"]
    if task_id == 2:
        desired_special_tokens.extend(["[LONG]", "[SHORT]"])
    vocab = tokenizer.get_vocab()
    tokens_to_add = [tok for tok in desired_special_tokens if tok not in vocab]
    if tokens_to_add:
        tokenizer.add_special_tokens({"additional_special_tokens": tokens_to_add})
    return tokenizer, len(tokens_to_add)


def make_loader(texts, labels, tokenizer, max_length, batch_size, shuffle=False):
    dataset = TextClassificationDataset(texts, tokenizer, max_length, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=torch.cuda.is_available())


def make_test_loader(texts, tokenizer, max_length, batch_size):
    dataset = TextClassificationDataset(texts, tokenizer, max_length, labels=None)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())


def move_batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}


def positive_f1(y_true, prob):
    pred = prob.argmax(axis=1)
    return f1_score(y_true, pred, pos_label=1)


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        outputs = model(**batch)
        probs = torch.softmax(outputs.logits, dim=-1)
        all_probs.append(probs.detach().cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def train_one_fold(task_id, fold, train_part, val_part, cfg, tokenizer, added_tokens):
    fold_dir = OUTPUT_ROOT / f"{PIPELINE_SLUG}_task{task_id}" / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=cfg["id2label"],
        label2id=cfg["label2id"],
    )
    if added_tokens > 0:
        model.resize_token_embeddings(len(tokenizer))
    if cfg.get("gradient_checkpointing", False):
        model.gradient_checkpointing_enable()
        model.config.use_cache = False
    model.to(DEVICE)

    train_loader = make_loader(
        train_part["text"].tolist(), train_part["label"].tolist(), tokenizer,
        cfg["max_length"], cfg["batch_size"], shuffle=True,
    )
    val_loader = make_loader(
        val_part["text"].tolist(), val_part["label"].tolist(), tokenizer,
        cfg["max_length"], cfg["batch_size"], shuffle=False,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
    )
    total_steps = max(1, len(train_loader) * cfg["epochs"])
    warmup_steps = int(total_steps * cfg["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

    best_f1 = -1.0
    best_epoch = -1
    history = []

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        losses = []
        progress = tqdm(train_loader, desc=f"Task {task_id} Fold {fold} Epoch {epoch}", leave=False)
        for batch in progress:
            batch = move_batch_to_device(batch)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                outputs = model(**batch)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))
            progress.set_postfix(loss=np.mean(losses))

        val_probs = predict_proba(model, val_loader)
        val_f1 = positive_f1(val_part["label"].values, val_probs)
        mean_loss = float(np.mean(losses)) if losses else 0.0
        history.append({"epoch": epoch, "train_loss": mean_loss, "val_f1": val_f1})
        print(f"Task {task_id} Fold {fold} | epoch={epoch} | loss={mean_loss:.4f} | val_f1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            model.save_pretrained(fold_dir)
            tokenizer.save_pretrained(fold_dir)

    # Reload best checkpoint for validation probabilities.
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    best_model = AutoModelForSequenceClassification.from_pretrained(fold_dir).to(DEVICE)
    val_probs = predict_proba(best_model, val_loader)

    del best_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return fold_dir, val_probs, best_f1, best_epoch, history


def ensemble_predict_from_dirs(fold_dirs, texts, max_length, batch_size):
    all_probs = []
    for fold_dir in fold_dirs:
        tokenizer = AutoTokenizer.from_pretrained(fold_dir, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(fold_dir).to(DEVICE)
        loader = make_test_loader(texts, tokenizer, max_length, batch_size)
        probs = predict_proba(model, loader)
        all_probs.append(probs)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return np.mean(all_probs, axis=0)


def run_transformer_task(task_id: int):
    set_seed(SEED)
    train_df, test_df, cfg = load_task_data(task_id)
    tokenizer, added_tokens = prepare_tokenizer(task_id)

    n_splits = 2 if FAST_DEBUG else N_FOLDS
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_probs = np.zeros((len(train_df), 2), dtype=np.float32)
    fold_dirs, fold_scores, fold_histories = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"]), start=1):
        train_part = train_df.iloc[tr_idx].reset_index(drop=True)
        val_part = train_df.iloc[va_idx].reset_index(drop=True)
        fold_dir, val_probs, best_f1, best_epoch, history = train_one_fold(
            task_id, fold, train_part, val_part, cfg, tokenizer, added_tokens
        )
        oof_probs[va_idx] = val_probs
        fold_dirs.append(fold_dir)
        fold_scores.append(best_f1)
        fold_histories.append(history)
        print(f"Best Task {task_id} Fold {fold}: F1={best_f1:.4f} at epoch {best_epoch}")

    oof_pred = oof_probs.argmax(axis=1)
    oof_f1 = f1_score(train_df["label"], oof_pred, pos_label=1)
    print("=" * 70)
    print(f"{PIPELINE_NAME} | Task {task_id} | OOF positive-class F1: {oof_f1:.4f}")
    print("Fold F1:", [round(x, 4) for x in fold_scores])
    print(classification_report(train_df["label"], oof_pred, target_names=[cfg["id2label"][0], cfg["id2label"][1]]))
    print("Confusion matrix:\n", confusion_matrix(train_df["label"], oof_pred))

    test_probs = ensemble_predict_from_dirs(
        fold_dirs,
        test_df["text"].tolist(),
        cfg["max_length"],
        cfg["batch_size"],
    )
    test_pred = test_probs.argmax(axis=1)
    pred_labels = [cfg["id2label"][int(x)] for x in test_pred]
    pred_path = OUTPUT_ROOT / f"{PIPELINE_SLUG}_task{task_id}_predictions.tsv"
    submission_df = pd.DataFrame({"id": test_df["id"], "tag": pred_labels})
    submission_df.to_csv(pred_path, sep="\t", index=False)
    print(f"Saved predictions: {pred_path}")
    display(submission_df.head())

    return {
        "task_id": task_id,
        "cfg": cfg,
        "train_df": train_df,
        "test_df": test_df,
        "fold_dirs": fold_dirs,
        "fold_scores": fold_scores,
        "fold_histories": fold_histories,
        "oof_probs": oof_probs,
        "oof_f1": oof_f1,
        "test_probs": test_probs,
        "prediction_path": pred_path,
    }

In [ ]:
# ============================================================
# Branch utilities for Pipeline 3
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score


def run_tfidf_branch(train_df, test_df, task_id, cfg):
    print("Running Task 1 TF-IDF branch...")
    n_splits = 2 if FAST_DEBUG else N_FOLDS
    y = train_df["label"].values
    oof = np.zeros(len(train_df), dtype=np.float32)
    test_accum = np.zeros(len(test_df), dtype=np.float32)

    vectorizer = FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
    ])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y), start=1):
        clf = Pipeline([
            ("tfidf", vectorizer),
            ("lr", LogisticRegression(C=4.0, max_iter=2000, class_weight="balanced", random_state=SEED)),
        ])
        clf.fit(train_df.iloc[tr_idx]["text"], y[tr_idx])
        oof[va_idx] = clf.predict_proba(train_df.iloc[va_idx]["text"])[:, 1]
        test_accum += clf.predict_proba(test_df["text"])[:, 1] / n_splits
        print(f"TF-IDF fold {fold} F1: {f1_score(y[va_idx], oof[va_idx] >= 0.5, pos_label=1):.4f}")
    print(f"TF-IDF OOF F1 @0.5: {f1_score(y, oof >= 0.5, pos_label=1):.4f}")
    return oof, test_accum


def stylometry_features(texts):
    rows = []
    for text in texts:
        text = clean_text(text)
        words = text.split()
        n_words = len(words)
        n_chars = len(text)
        unique_words = len(set(w.lower() for w in words))
        punct_count = sum(1 for ch in text if ch in ".,;:!?¡¿")
        upper_count = sum(1 for ch in text if ch.isupper())
        digit_count = sum(1 for ch in text if ch.isdigit())
        rows.append({
            "n_chars": n_chars,
            "n_words": n_words,
            "avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
            "type_token_ratio": unique_words / max(1, n_words),
            "punct_ratio": punct_count / max(1, n_chars),
            "upper_ratio": upper_count / max(1, n_chars),
            "digit_ratio": digit_count / max(1, n_chars),
            "excl_count": text.count("!") + text.count("¡"),
            "ques_count": text.count("?") + text.count("¿"),
            "comma_count": text.count(","),
            "long_hint": 1.0 if n_words >= 18 else 0.0,
        })
    return pd.DataFrame(rows)


@torch.no_grad()
def compute_perplexity_features(texts, batch_size=8, max_length=128):
    defaults = pd.DataFrame({"mean_nll": np.zeros(len(texts)), "perplexity": np.ones(len(texts))})
    if not USE_PERPLEXITY:
        return defaults
    try:
        from transformers import AutoModelForCausalLM
        ppl_tokenizer = AutoTokenizer.from_pretrained(PERPLEXITY_MODEL_NAME)
        ppl_model = AutoModelForCausalLM.from_pretrained(PERPLEXITY_MODEL_NAME).to(DEVICE)
        ppl_model.eval()
        if ppl_tokenizer.pad_token is None:
            ppl_tokenizer.pad_token = ppl_tokenizer.eos_token
        mean_nlls, ppls = [], []
        for start in tqdm(range(0, len(texts), batch_size), desc="Perplexity"):
            batch_texts = [clean_text(x) for x in texts[start:start + batch_size]]
            enc = ppl_tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=max_length).to(DEVICE)
            labels = enc["input_ids"].clone()
            outputs = ppl_model(**enc, labels=labels)
            logits = outputs.logits[:, :-1, :].contiguous()
            target = labels[:, 1:].contiguous()
            mask = enc["attention_mask"][:, 1:].contiguous()
            loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
            token_loss = loss_fct(logits.view(-1, logits.size(-1)), target.view(-1)).view(target.size())
            lengths = mask.sum(dim=1).clamp(min=1)
            sent_nll = (token_loss * mask).sum(dim=1) / lengths
            sent_ppl = torch.exp(sent_nll.clamp(max=20))
            mean_nlls.extend(sent_nll.detach().cpu().numpy().tolist())
            ppls.extend(sent_ppl.detach().cpu().numpy().tolist())
        del ppl_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return pd.DataFrame({"mean_nll": mean_nlls, "perplexity": ppls})
    except Exception as exc:
        print("Perplexity branch failed; falling back to zero/default features.")
        print("Reason:", repr(exc))
        return defaults


def style_feature_matrix(df, use_joke_only=True):
    # For Task 2, use joke-only style because the label is about the joke author.
    if use_joke_only and "joke" in df.columns:
        base_texts = df["joke"].fillna("").tolist()
    else:
        base_texts = df["text"].fillna("").tolist()
    style = stylometry_features(base_texts)
    ppl = compute_perplexity_features(base_texts)
    return pd.concat([style.reset_index(drop=True), ppl.reset_index(drop=True)], axis=1)


def run_style_branch(train_df, test_df, task_id, cfg):
    print("Running Task 2 perplexity/stylometry branch...")
    n_splits = 2 if FAST_DEBUG else N_FOLDS
    y = train_df["label"].values
    X_train = style_feature_matrix(train_df, use_joke_only=True)
    X_test = style_feature_matrix(test_df, use_joke_only=True)
    oof = np.zeros(len(train_df), dtype=np.float32)
    test_accum = np.zeros(len(test_df), dtype=np.float32)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y), start=1):
        clf = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("gbdt", GradientBoostingClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=3,
                min_samples_leaf=10,
                random_state=SEED + fold,
            )),
        ])
        clf.fit(X_train.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = clf.predict_proba(X_train.iloc[va_idx])[:, 1]
        test_accum += clf.predict_proba(X_test)[:, 1] / n_splits
        print(f"Style fold {fold} F1: {f1_score(y[va_idx], oof[va_idx] >= 0.5, pos_label=1):.4f}")
    print(f"Style OOF F1 @0.5: {f1_score(y, oof >= 0.5, pos_label=1):.4f}")
    return oof, test_accum


def tune_threshold(y_true, prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = [f1_score(y_true, prob >= t, pos_label=1) for t in thresholds]
    best_idx = int(np.argmax(scores))
    return float(thresholds[best_idx]), float(scores[best_idx])


def stack_two_branches(y, train_prob_a, train_prob_b, test_prob_a, test_prob_b):
    X_stack = np.column_stack([train_prob_a, train_prob_b])
    X_test_stack = np.column_stack([test_prob_a, test_prob_b])
    meta = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0, random_state=SEED)
    meta.fit(X_stack, y)
    oof_stack = meta.predict_proba(X_stack)[:, 1]
    test_stack = meta.predict_proba(X_test_stack)[:, 1]
    threshold, best_f1 = tune_threshold(y, oof_stack)
    print(f"Stacked OOF best F1={best_f1:.4f} at threshold={threshold:.3f}")
    return oof_stack, test_stack, threshold, best_f1

In [ ]:
# ============================================================
# Run Pipeline 3 on both subtasks
# ============================================================
PIPELINE3_RESULTS = {}
for task_id in TASKS_TO_RUN:
    print("\n" + "#" * 80)
    print(f"Pipeline 3 on Task {task_id}")
    print("#" * 80)

    transformer_result = run_transformer_task(task_id)
    train_df = transformer_result["train_df"]
    test_df = transformer_result["test_df"]
    cfg = transformer_result["cfg"]
    y = train_df["label"].values
    transformer_oof = transformer_result["oof_probs"][:, 1]
    transformer_test = transformer_result["test_probs"][:, 1]

    if task_id == 1:
        aux_oof, aux_test = run_tfidf_branch(train_df, test_df, task_id, cfg)
        aux_name = "tfidf"
    elif task_id == 2:
        aux_oof, aux_test = run_style_branch(train_df, test_df, task_id, cfg)
        aux_name = "perplexity_stylometry"
    else:
        raise ValueError(task_id)

    stack_oof, stack_test, threshold, best_f1 = stack_two_branches(
        y, transformer_oof, aux_oof, transformer_test, aux_test
    )
    final_pred_ids = (stack_test >= threshold).astype(int)
    final_labels = [cfg["id2label"][int(x)] for x in final_pred_ids]

    pred_path = OUTPUT_ROOT / f"{PIPELINE_SLUG}_task{task_id}_predictions.tsv"
    submission_df = pd.DataFrame({"id": test_df["id"], "tag": final_labels})
    submission_df.to_csv(pred_path, sep="\t", index=False)
    print(f"Saved stacked predictions: {pred_path}")
    display(submission_df.head())

    PIPELINE3_RESULTS[task_id] = {
        "task_id": task_id,
        "aux_branch": aux_name,
        "threshold": threshold,
        "oof_f1": best_f1,
        "prediction_path": pred_path,
    }

zip_path = OUTPUT_ROOT / f"{PIPELINE_SLUG}_submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for task_id, result in PIPELINE3_RESULTS.items():
        zf.write(result["prediction_path"], arcname=f"task{task_id}.tsv")
print(f"Created submission ZIP: {zip_path}")